Merkle trees are used in Bitcoin because:

1. It allows you check if a transaction is included in the blockchain, yourself. 
2. And, you can do this with less data.. you don't have to download all the blocks & transactions (800GB on Nov 2026) and check each one by one for inclusion. Instead, you can download just the blocks with headers (~78MB), and prove that a transaction exists in a block with the help of merkle trees.

To make Merkle trees work, we need to have:
1. An ordered list.
2. A hash function.

In Bitcoin, the ordered list is the list of transactions included in the block, and the hash function is SHA256. The basic idea is to hash a pair of transactions, to create the merkle parent.

In [5]:
import math
import sys
from importlib import reload
from io import BytesIO
from pathlib import Path

# Use the functions from Programming Bitcoin, Chapter 11.
chapter_dir = str(Path("../programmingbitcoin/book/code-ch11").resolve())
if chapter_dir not in sys.path:
    sys.path.insert(0, chapter_dir)

import block
import ecc
import helper
import merkleblock
import network
import script
import tx

from block import Block
from helper import (
    bytes_to_bit_field,
    hash256,
    little_endian_to_int,
    merkle_parent,
    merkle_parent_level,
    merkle_root,
    read_varint,
    run,
)
from merkleblock import MerkleBlock, MerkleTree
from network import GetHeadersMessage, HeadersMessage, SimpleNode
from tx import Tx, TxFetcher

tr_1 = bytes.fromhex('c117ea8ec828342f4dfb0ad6bd140e03a50720ece40169ee38bdc15d9eb64cf5')
tr_2 = bytes.fromhex('c131474164b412e3406696da1ee20ab0fc9bf41c8f05fa8ceea7a08d672d7cc5')
parent = hash256(tr_1 + tr_2)
print(parent.hex())

8b30c5ba100f6f2e5ad1e2a742e5020491240f8eb514fe97c713c31718ad7ecd


Since the parent is a combination of two children, the parent row will always have half the number of hashes of the children row. We can then continue hashing the merkle parent rows, until we are left with just one single hash in a row, called the merkle root. The merkle root is what is included in the block header, and a light client can prove or verify that a transaction is included by requesting only the sibling hashes of each row from the full node.

This is an example of a merkle tree with 10 items:

In [19]:
total = 10
max_depth = math.ceil(math.log(total, 2))
merkle_tree = []
for depth in range(max_depth + 1):
    re = 2**(max_depth - depth)
    num_items = math.ceil(total / 2**(max_depth - depth))
    level_hashes = [None] * num_items
    merkle_tree.append(level_hashes)
for level in merkle_tree:
    print(level)

[None]
[None, None]
[None, None, None]
[None, None, None, None, None]
[None, None, None, None, None, None, None, None, None, None]


In [21]:
hex_hashes = [
    "9745f7173ef14ee4155722d1cbf13304339fd00d900b759c6f9d58579b5765fb",
    "5573c8ede34936c29cdfdfe743f7f5fdfbd4f54ba0705259e62f39917065cb9b",
    "82a02ecbb6623b4274dfcab82b336dc017a27136e08521091e443e62582e8f05",
    "507ccae5ed9b340363a0e6d765af148be9cb1c8766ccc922f83e4ae681658308",
    "a7a4aec28e7162e1e9ef33dfa30f0bc0526e6cf4b11a576f6c5de58593898330",
    "bb6267664bd833fd9fc82582853ab144fece26b7a8a5bf328f8a059445b59add",
    "ea6d7ac1ee77fbacee58fc717b990c4fcccf1b19af43103c090f601677fd8836",
    "457743861de496c429912558a106b810b0507975a49773228aa788df40730d41",
    "7688029288efc9e9a0011c960a6ed9e5466581abf3e3a6c26ee317461add619a",
    "b1ae7f15836cb2286cdd4e2c37bf9bb7da0a2846d06867a429f654b2e7f383c9",
]
tree = MerkleTree(len(hex_hashes))
last_level = len(tree.nodes) - 1

tree.nodes[last_level] = [bytes.fromhex(h) for h in hex_hashes]

for depth in range(last_level, 0, -1):
    tree.nodes[depth - 1] = merkle_parent_level(
        tree.nodes[depth].copy()
    )

print(tree)

*14c4eea1.*
6382df3f..., a8cef966...
3ba6c080..., 8e894862..., 8c65546c...
272945ec..., 9a38d037..., 4a64abd9..., ec7c95e1..., 3b67006c...
9745f717..., 5573c8ed..., 82a02ecb..., 507ccae5..., a7a4aec2..., bb626766..., ea6d7ac1..., 45774386..., 76880292..., b1ae7f15...


Next, let's get a Merkle proof via RPC. We can then copy it and use it in the notebook.

##### 1. Choose a recent block and read its header

```bash
proof_tip=$(/usr/local/bin/bitcoin-cli getblockcount)

proof_height=$((proof_tip - RANDOM % 20))

proof_hash=$(/usr/local/bin/bitcoin-cli getblockhash "$proof_height")

/usr/local/bin/bitcoin-cli getblockheader "$proof_hash"
```

```text
Block height: 890946
Block hash: 000000000000000000020eb7a008cc27d19df355d8695f029c08a156c7253fdc
Merkle root: 5d9e3af145efd681e5b79720d67600bd6172a238aa03ce5e31dee48909beb938
Transaction count: 139
```

The header contains the Merkle root. Reading the header does not download the full block.

#### 2. Save the full block data and select a transaction

The `2` requests details for every transaction. This command reads the block from the local node and saves the returned JSON data.

```bash
/usr/local/bin/bitcoin-cli getblock "000000000000000000020eb7a008cc27d19df355d8695f029c08a156c7253fdc" 2 |
/usr/bin/python3 -c '
import json
import sys
from pathlib import Path

block = json.load(sys.stdin)

path = Path.home() / "Downloads" / "bitcoin-block-890946.json"
path.write_text(json.dumps(block, indent=2) + "\n")

print("Saved full block data to:", path)
print("Block height:", block["height"])
print("Transaction count:", block["nTx"])
print("Transaction ID:", block["tx"][0]["txid"])
'
```

Output:

```text
Saved full block data to: /Users/arjunrajkumar/Downloads/bitcoin-block-890946.json
Block height: 890946
Transaction count: 139
Transaction ID: 89660fb69df0ed4b2d1655961948eecf0c358d37d52408d2e325bcd070ce50b3
```

#### 3. Request the proof for that transaction

Run this in Terminal. `gettxoutproof` returns the proof as hexadecimal text. The block hash tells the node which block to use. `tee` prints the proof and saves a copy.

```bash
/usr/local/bin/bitcoin-cli gettxoutproof \
  '["89660fb69df0ed4b2d1655961948eecf0c358d37d52408d2e325bcd070ce50b3"]' \
  "000000000000000000020eb7a008cc27d19df355d8695f029c08a156c7253fdc" \
  | tee "$HOME/Downloads/bitcoin-proof-890946.hex"
```

The returned proof contains a block header, the total transaction count, proof hashes, and flag bytes. The complete proof text is in the next cell. That cell parses the proof into fields; parsing alone does not verify transaction inclusion.

Command references: [getblockheader](https://bitcoincore.org/en/doc/24.0.0/rpc/blockchain/getblockheader/), [getblock](https://bitcoincore.org/en/doc/24.0.0/rpc/blockchain/getblock/), and [gettxoutproof](https://bitcoincore.org/en/doc/24.0.0/rpc/blockchain/gettxoutproof/).


In [27]:
import sys
from io import BytesIO
from pathlib import Path

# Use the Chapter 11 parser from the local book checkout.
chapter_dir = str(Path("../programmingbitcoin/book/code-ch11").resolve())
if chapter_dir not in sys.path:
    sys.path.insert(0, chapter_dir)

from importlib import reload
import merkleblock

merkleblock = reload(merkleblock)
MerkleBlock = merkleblock.MerkleBlock

# Keep the selected block and transaction for the later proof check.
block_hash = "000000000000000000020eb7a008cc27d19df355d8695f029c08a156c7253fdc"
txid = "89660fb69df0ed4b2d1655961948eecf0c358d37d52408d2e325bcd070ce50b3"
expected_merkle_root = "5d9e3af145efd681e5b79720d67600bd6172a238aa03ce5e31dee48909beb938"

# Proof returned by bitcoin-cli gettxoutproof.
proof_hex = (
    "00a00e20c31a2ffc2e001d2979da36bf4ff2b2af262d0cded6cc0100000000000000000038b9be0989e4de315ece03aa"
    "38a27261bd0076d62097b7e581d6ef45f13a9e5d0f53f0676c7902176b353f058b00000009b350ce70d0bc25e3d20824"
    "d5378d350ccfee48199655162d4bedf09db60f668921d4fa09f669e70286c7efe6d8aac76b582c9bf2dbf7b653fde443"
    "a8dd889602a97ea4ca6ce9b6f563dd4373f5bd12213ebc2dd370bea4709d00da19471f06834182c9475ae0045061d04a"
    "66d8812c2f1f519ceb587812384eb92a60f6bd6712b456e6af0c71a16824fbcf784d5822630798bb102ad015369b28e7"
    "992a5240a8b933fc9756ab501e49d0c966a701d2358b43508f7f2b47bd79a37c4e94e8aac7d4bf5e33741a4bd36ffb2d"
    "fba368e0f498f4192a56af96f03d657a79e27f463c2e000f3fb5c9d955570e27209d2bc43a92ba4ab57de101d7ce771f"
    "cfa5739a251329de31658e6821e48e4051efe787a5cac089f90613188a2d354c4f4569f2e703ff0100"
)

# Parse the saved text. This does not connect to Bitcoin Core.
proof_bytes = bytes.fromhex(proof_hex)
proof_stream = BytesIO(proof_bytes)
merkle_block = MerkleBlock.parse(proof_stream)

print("Proof size (bytes):", len(proof_bytes))
print("Transaction count:", merkle_block.total)
print("Merkle root:", merkle_block.merkle_root.hex())
print("Proof hash count:", len(merkle_block.hashes))
print("Flags:", merkle_block.flags.hex())
print("Unread bytes:", len(proof_stream.read()))

for index, proof_hash in enumerate(merkle_block.hashes):
    print(f"Hash {index}:", proof_hash.hex())


Proof size (bytes): 377
Transaction count: 139
Merkle root: 5d9e3af145efd681e5b79720d67600bd6172a238aa03ce5e31dee48909beb938
Proof hash count: 9
Flags: ff0100
Unread bytes: 0
Hash 0: 89660fb69df0ed4b2d1655961948eecf0c358d37d52408d2e325bcd070ce50b3
Hash 1: 029688dda843e4fd53b6f7dbf29b2c586bc7aad8e6efc78602e769f609fad421
Hash 2: 83061f4719da009d70a4be70d32dbc3e2112bdf57343dd63f5b6e96ccaa47ea9
Hash 3: 1267bdf6602ab94e38127858eb9c511f2f2c81d8664ad0615004e05a47c98241
Hash 4: a840522a99e7289b3615d02a10bb98076322584d78cffb2468a1710cafe656b4
Hash 5: c7aae8944e7ca379bd472b7f8f50438b35d201a766c9d0491e50ab5697fc33b9
Hash 6: 3c467fe2797a653df096af562a19f498f4e068a3fb2dfb6fd34b1a74335ebfd4
Hash 7: 259a73a5cf1f77ced701e17db54aba923ac42b9d20270e5755d9c9b53f0f002e
Hash 8: e7f269454f4c352d8a181306f989c0caa587e7ef51408ee421688e6531de2913


In [28]:
from helper import hash256

header_hash = hash256(proof_bytes[:80])[::-1].hex()

print("Correct block:", header_hash == block_hash)
print(
    "Correct Merkle root:",
    merkle_block.merkle_root.hex() == expected_merkle_root,
)
print("Proof rebuilds the root:", merkle_block.is_valid())

Correct block: True
Correct Merkle root: True
Proof rebuilds the root: True
